# Deep Q-Networks (DQN) - From Tabular to Deep RL

[![Open In Colab](https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/rl-deep-q-networks.ipynb)](https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/rl-deep-q-networks.ipynb)

This notebook bridges tabular reinforcement learning to deep RL by introducing **Deep Q-Networks (DQN)**, the breakthrough algorithm that enabled agents to learn from high-dimensional observations like images.

## 1. Introduction

### What We'll Learn

- Why tabular Q-learning fails for complex environments
- How neural networks can approximate Q-values
- The **experience replay** mechanism and why it's crucial
- The **target network** trick for stable training
- Complete DQN implementation on CartPole

### The Big Picture

**Tabular Q-learning** stores Q(s,a) in a table - one entry per state-action pair. This works for small state spaces (e.g., grid worlds with 100 states), but breaks down when:

- **State space is huge**: Atari games have 210×160×3 = 100,800 pixel values → ~256^100,800 possible states!
- **Continuous states**: CartPole has continuous position/velocity → infinitely many states
- **Generalization needed**: Similar states should have similar values

**Solution**: Use a neural network $Q(s,a;\theta)$ to approximate Q-values instead of storing them in a table. This is **function approximation**.

## 2. Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from collections import deque, namedtuple
import random
import gymnasium as gym
from IPython.display import clear_output
from aiml_notebooks import get_device, set_seed

In [ ]:
# Set random seed for reproducibility
set_seed(42)

# Device configuration
device = get_device()
print(f"Using device: {device}")

## 3. The CartPole Environment

**CartPole-v1** is a classic control task where you balance a pole on a moving cart.

### State Space (4 continuous values)
1. Cart position: -4.8 to 4.8
2. Cart velocity: -∞ to ∞
3. Pole angle: -0.418 to 0.418 radians (~24°)
4. Pole angular velocity: -∞ to ∞

### Action Space (2 discrete actions)
- 0: Push cart left
- 1: Push cart right

### Reward
- +1 for every timestep the pole stays upright
- Episode ends if pole angle > 12° or cart moves > 2.4 units from center
- **Goal**: Survive 500 timesteps (maximum episode length)

### Why This is Hard for Tabular Q-Learning

The state space is **continuous** → infinitely many states! Even if we discretize (e.g., 10 bins per dimension), we'd need 10^4 = 10,000 table entries. For Atari, we'd need trillions.

In [ ]:
# Create CartPole environment
env = gym.make('CartPole-v1')

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

print(f"State dimension: {state_dim}")
print(f"Action dimension: {action_dim}")
print(f"State space: {env.observation_space}")
print(f"Action space: {env.action_space}")

Let's see what a random episode looks like:

In [ ]:
# Run a random episode
state, _ = env.reset(seed=42)
total_reward = 0
done = False
steps = 0

print("Initial state:", state)

while not done and steps < 50:
    action = env.action_space.sample()  # Random action
    next_state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    total_reward += reward
    steps += 1
    state = next_state

print(f"\nRandom policy achieved {total_reward} reward in {steps} steps")
print("A good policy should reach 500 steps!")

## 4. Neural Network Q-Function

### From Table to Function

**Tabular Q-learning**: Q-table with shape (num_states, num_actions)

**Deep Q-Network**: Neural network $Q(s,a;\theta)$ that takes state $s$ as input and outputs Q-values for all actions.

```
State (4 values)  →  [Linear] → [ReLU] → [Linear] → [ReLU] → [Linear]  →  Q-values (2 values)
[pos, vel, ...]      Hidden 128         Hidden 128                        [Q(s,left), Q(s,right)]
```

### Why Output All Actions?

For action selection, we need $\max_a Q(s,a)$. Computing Q for all actions in one forward pass is efficient!

In [ ]:
class QNetwork(nn.Module):
    """Neural network for approximating Q-values."""
    
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, action_dim)
    
    def forward(self, state):
        """Forward pass: state → Q-values for all actions."""
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        q_values = self.fc3(x)  # No activation - Q-values can be any real number
        return q_values

Let's test the network:

In [ ]:
# Create a test network
q_net = QNetwork(state_dim, action_dim).to(device)

# Test with random state
test_state = torch.randn(1, state_dim).to(device)
q_values = q_net(test_state)

print(f"Input state shape: {test_state.shape}")
print(f"Output Q-values shape: {q_values.shape}")
print(f"Q-values: {q_values}")
print(f"Best action: {q_values.argmax().item()}")

## 5. Experience Replay Buffer

### The Problem: Correlated Samples

In online Q-learning, we learn from consecutive experiences:

```
t=1: (s₁, a₁, r₁, s₂)  ← highly correlated!
t=2: (s₂, a₂, r₂, s₃)  ← s₂ appears in both
t=3: (s₃, a₃, r₃, s₄)  ← sequential experiences
```

**Problem**: Neural networks trained on correlated data can:
- **Forget quickly** (catastrophic forgetting)
- **Overfit to recent experiences**
- **Oscillate** (unstable learning)

### The Solution: Experience Replay

Store experiences in a **replay buffer** and sample **random mini-batches** for training.

```
Buffer: [(s₁,a₁,r₁,s₂), (s₂,a₂,r₂,s₃), ..., (s₉₉,a₉₉,r₉₉,s₁₀₀)]
                           ↓
Random sample: [(s₃₇,a₃₇,r₃₇,s₃₈), (s₅,a₅,r₅,s₆), (s₈₂,a₈₂,r₈₂,s₈₃), ...]
```

**Benefits**:
1. **Breaks correlation**: Random sampling → independent samples
2. **Data efficiency**: Reuse old experiences multiple times
3. **Stable learning**: Smooth out the training signal

In [ ]:
# Named tuple for storing transitions
Transition = namedtuple('Transition', ['state', 'action', 'reward', 'next_state', 'done'])

class ReplayBuffer:
    """Experience replay buffer for storing and sampling transitions."""
    
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        """Store a transition."""
        self.buffer.append(Transition(state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        """Sample a random batch of transitions."""
        batch = random.sample(self.buffer, batch_size)
        
        # Unzip the batch
        states = torch.FloatTensor(np.array([t.state for t in batch]))
        actions = torch.LongTensor([t.action for t in batch])
        rewards = torch.FloatTensor([t.reward for t in batch])
        next_states = torch.FloatTensor(np.array([t.next_state for t in batch]))
        dones = torch.FloatTensor([t.done for t in batch])
        
        return states, actions, rewards, next_states, dones
    
    def __len__(self):
        return len(self.buffer)

Let's test the replay buffer:

In [ ]:
# Test replay buffer
buffer = ReplayBuffer(capacity=100)

# Add some fake transitions
for i in range(50):
    state = np.random.randn(4)
    action = np.random.randint(0, 2)
    reward = 1.0
    next_state = np.random.randn(4)
    done = False
    buffer.push(state, action, reward, next_state, done)

print(f"Buffer size: {len(buffer)}")

# Sample a batch
states, actions, rewards, next_states, dones = buffer.sample(batch_size=8)

print(f"\nBatch shapes:")
print(f"  States: {states.shape}")
print(f"  Actions: {actions.shape}")
print(f"  Rewards: {rewards.shape}")
print(f"  Next states: {next_states.shape}")
print(f"  Dones: {dones.shape}")

## 6. Target Network

### The Problem: Moving Target

In Q-learning, the update rule is:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma \max_{a'} Q(s',a') - Q(s,a) \right]$$

The **TD target** is: $r + \gamma \max_{a'} Q(s',a')$

**Problem**: We use the same network $Q$ to compute both:
1. Current Q-value: $Q(s,a)$
2. Target Q-value: $\max_{a'} Q(s',a')$

When we update the network, **both the prediction AND the target change**! This creates a moving target:

```
Before update: Q(s,a) = 5,  Target = 10  →  Loss = (5-10)² = 25
After update:  Q(s,a) = 7,  Target = 12  →  Target moved!
```

This leads to **unstable training** and **divergence**.

### The Solution: Target Network

Maintain **two networks**:
1. **Q-network** ($\theta$): Updated every step
2. **Target network** ($\theta^-$): Frozen copy, updated every C steps

$$\text{TD target} = r + \gamma \max_{a'} Q(s',a'; \theta^-)$$

The target network provides a **stable target** for several training steps before being updated.

**Update schedule**:
```
Every step:     θ ← θ - α∇Loss(θ)     (train Q-network)
Every C steps:  θ⁻ ← θ                (copy Q-network to target network)
```

Let's visualize the difference between using the same network vs. target network:

In [ ]:
# Demonstrate target network concept
q_network = QNetwork(state_dim, action_dim).to(device)
target_network = QNetwork(state_dim, action_dim).to(device)

# Initially copy weights
target_network.load_state_dict(q_network.state_dict())

# Test state
test_state = torch.randn(1, state_dim).to(device)

# Before training
q_vals_before = q_network(test_state)
target_vals_before = target_network(test_state)

print("Before any updates:")
print(f"Q-network output: {q_vals_before.detach().cpu().numpy()}")
print(f"Target network output: {target_vals_before.detach().cpu().numpy()}")
print(f"Are they equal? {torch.allclose(q_vals_before, target_vals_before)}")

# Simulate one gradient update on Q-network
optimizer = optim.Adam(q_network.parameters(), lr=0.01)
loss = q_vals_before.pow(2).sum()  # Dummy loss
optimizer.zero_grad()
loss.backward()
optimizer.step()

# After training
q_vals_after = q_network(test_state)
target_vals_after = target_network(test_state)

print("\nAfter one Q-network update:")
print(f"Q-network output: {q_vals_after.detach().cpu().numpy()}")
print(f"Target network output: {target_vals_after.detach().cpu().numpy()}")
print(f"Are they equal? {torch.allclose(q_vals_after, target_vals_after)}")
print("\n→ Target network stays frozen while Q-network updates!")

## 7. DQN Agent Implementation

Now we'll put it all together into a complete DQN agent.

### DQN Algorithm

```
Initialize Q-network Q(s,a;θ) and target network Q(s,a;θ⁻)
Initialize replay buffer D

For each episode:
    For each step:
        1. Select action: ε-greedy based on Q(s,a;θ)
        2. Execute action, observe r, s'
        3. Store (s,a,r,s') in D
        4. Sample random batch from D
        5. Compute TD target: y = r + γ·max_a' Q(s',a';θ⁻)
        6. Update Q-network: minimize (Q(s,a;θ) - y)²
        7. Every C steps: θ⁻ ← θ
```

### Key Hyperparameters

- **Learning rate** (α): Step size for gradient updates (e.g., 0.001)
- **Discount factor** (γ): How much to value future rewards (e.g., 0.99)
- **Epsilon** (ε): Exploration rate for ε-greedy (e.g., start 1.0, decay to 0.01)
- **Batch size**: Number of experiences per training step (e.g., 64)
- **Buffer size**: Replay buffer capacity (e.g., 10000)
- **Target update frequency**: How often to copy θ → θ⁻ (e.g., every 100 steps)

In [ ]:
class DQNAgent:
    """Deep Q-Network agent."""
    
    def __init__(
        self,
        state_dim,
        action_dim,
        device,
        lr=1e-3,
        gamma=0.99,
        epsilon_start=1.0,
        epsilon_end=0.01,
        epsilon_decay=0.995,
        buffer_size=10000,
        batch_size=64,
        target_update_freq=100
    ):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.device = device
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        
        # Epsilon-greedy parameters
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        
        # Networks
        self.q_network = QNetwork(state_dim, action_dim).to(device)
        self.target_network = QNetwork(state_dim, action_dim).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        
        # Optimizer
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
        
        # Replay buffer
        self.replay_buffer = ReplayBuffer(capacity=buffer_size)
        
        # Tracking
        self.steps = 0
        self.losses = []
    
    def select_action(self, state, greedy=False):
        """Select action using epsilon-greedy policy."""
        if not greedy and random.random() < self.epsilon:
            # Explore: random action
            return random.randint(0, self.action_dim - 1)
        else:
            # Exploit: best action according to Q-network
            with torch.no_grad():
                state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
                q_values = self.q_network(state_tensor)
                return q_values.argmax().item()
    
    def store_transition(self, state, action, reward, next_state, done):
        """Store transition in replay buffer."""
        self.replay_buffer.push(state, action, reward, next_state, done)
    
    def train_step(self):
        """Perform one training step."""
        if len(self.replay_buffer) < self.batch_size:
            return None
        
        # Sample batch
        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)
        
        states = states.to(self.device)
        actions = actions.to(self.device)
        rewards = rewards.to(self.device)
        next_states = next_states.to(self.device)
        dones = dones.to(self.device)
        
        # Compute current Q-values
        current_q_values = self.q_network(states).gather(1, actions.unsqueeze(1)).squeeze()
        
        # Compute target Q-values using target network
        with torch.no_grad():
            next_q_values = self.target_network(next_states).max(1)[0]
            target_q_values = rewards + (1 - dones) * self.gamma * next_q_values
        
        # Compute loss
        loss = F.mse_loss(current_q_values, target_q_values)
        
        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        # Update target network
        self.steps += 1
        if self.steps % self.target_update_freq == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())
        
        # Decay epsilon
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
        
        self.losses.append(loss.item())
        return loss.item()

## 8. Training Loop

Let's train the DQN agent on CartPole!

In [ ]:
def train_dqn(agent, env, num_episodes=500, print_every=50):
    """Train DQN agent."""
    episode_rewards = []
    episode_lengths = []
    
    for episode in range(num_episodes):
        state, _ = env.reset()
        episode_reward = 0
        episode_length = 0
        done = False
        
        while not done:
            # Select and perform action
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # Store transition
            agent.store_transition(state, action, reward, next_state, done)
            
            # Train agent
            agent.train_step()
            
            state = next_state
            episode_reward += reward
            episode_length += 1
        
        episode_rewards.append(episode_reward)
        episode_lengths.append(episode_length)
        
        # Print progress
        if (episode + 1) % print_every == 0:
            avg_reward = np.mean(episode_rewards[-print_every:])
            avg_length = np.mean(episode_lengths[-print_every:])
            print(f"Episode {episode+1}/{num_episodes} | "
                  f"Avg Reward: {avg_reward:.2f} | "
                  f"Avg Length: {avg_length:.2f} | "
                  f"Epsilon: {agent.epsilon:.3f}")
    
    return episode_rewards, episode_lengths

Create and train the agent:

In [ ]:
# Create agent
agent = DQNAgent(
    state_dim=state_dim,
    action_dim=action_dim,
    device=device,
    lr=1e-3,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.01,
    epsilon_decay=0.995,
    buffer_size=10000,
    batch_size=64,
    target_update_freq=100
)

print("Training DQN agent...\n")
episode_rewards, episode_lengths = train_dqn(agent, env, num_episodes=200, print_every=50)

## 9. Evaluation and Visualization

Let's see how well our agent learned!

In [ ]:
def evaluate_agent(agent, env, num_episodes=10):
    """Evaluate trained agent without exploration."""
    rewards = []
    
    for _ in range(num_episodes):
        state, _ = env.reset()
        episode_reward = 0
        done = False
        
        while not done:
            action = agent.select_action(state, greedy=True)  # No exploration
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            episode_reward += reward
            state = next_state
        
        rewards.append(episode_reward)
    
    return rewards

# Evaluate
eval_rewards = evaluate_agent(agent, env, num_episodes=10)
print(f"Evaluation over 10 episodes:")
print(f"  Mean reward: {np.mean(eval_rewards):.2f}")
print(f"  Std reward: {np.std(eval_rewards):.2f}")
print(f"  Min reward: {np.min(eval_rewards):.2f}")
print(f"  Max reward: {np.max(eval_rewards):.2f}")
print(f"\nGoal is 500 - we {'SUCCEEDED!' if np.mean(eval_rewards) >= 475 else 'need more training.'}")

### Training Curves

Let's visualize the learning progress:

In [ ]:
# Plot learning curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Episode rewards
axes[0, 0].plot(episode_rewards, alpha=0.3, color='blue')
axes[0, 0].plot(np.convolve(episode_rewards, np.ones(20)/20, mode='valid'), color='blue', linewidth=2, label='Moving avg (20)')
axes[0, 0].axhline(y=500, color='green', linestyle='--', alpha=0.5, label='Goal')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Total Reward')
axes[0, 0].set_title('Episode Rewards During Training')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Episode lengths
axes[0, 1].plot(episode_lengths, alpha=0.3, color='orange')
axes[0, 1].plot(np.convolve(episode_lengths, np.ones(20)/20, mode='valid'), color='orange', linewidth=2, label='Moving avg (20)')
axes[0, 1].axhline(y=500, color='green', linestyle='--', alpha=0.5, label='Max length')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Episode Length')
axes[0, 1].set_title('Episode Lengths During Training')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Training loss
if agent.losses:
    axes[1, 0].plot(agent.losses, alpha=0.3, color='red')
    if len(agent.losses) > 100:
        axes[1, 0].plot(np.convolve(agent.losses, np.ones(100)/100, mode='valid'), 
                       color='red', linewidth=2, label='Moving avg (100)')
    axes[1, 0].set_xlabel('Training Step')
    axes[1, 0].set_ylabel('TD Loss')
    axes[1, 0].set_title('Training Loss (MSE)')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

# Evaluation rewards
axes[1, 1].bar(range(len(eval_rewards)), eval_rewards, color='green', alpha=0.7)
axes[1, 1].axhline(y=np.mean(eval_rewards), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(eval_rewards):.1f}')
axes[1, 1].axhline(y=500, color='blue', linestyle='--', alpha=0.5, label='Goal: 500')
axes[1, 1].set_xlabel('Evaluation Episode')
axes[1, 1].set_ylabel('Total Reward')
axes[1, 1].set_title('Evaluation Performance (Greedy Policy)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### Understanding the Learning Curves

**Top-left (Episode Rewards)**: Shows how total reward per episode increases over training. Initially low (random policy), then improves as the agent learns.

**Top-right (Episode Lengths)**: In CartPole, reward = episode length. Longer episodes mean better balancing.

**Bottom-left (Training Loss)**: TD loss measures prediction error. High initially (poor estimates), decreases as Q-values improve, then may increase slightly as the agent explores harder states.

**Bottom-right (Evaluation)**: Final performance with greedy policy (no exploration). Should be consistently high if training succeeded.

## 10. Ablation Studies

Let's understand what makes DQN work by removing key components.

### Ablation 1: No Experience Replay

Train on consecutive experiences instead of random samples from the buffer.

In [ ]:
class DQNAgentNoReplay(DQNAgent):
    """DQN without experience replay - trains on immediate experiences."""
    
    def train_step_immediate(self, state, action, reward, next_state, done):
        """Train immediately on current transition."""
        # Convert to tensors
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        action = torch.LongTensor([action]).to(self.device)
        reward = torch.FloatTensor([reward]).to(self.device)
        next_state = torch.FloatTensor(next_state).unsqueeze(0).to(self.device)
        done = torch.FloatTensor([done]).to(self.device)
        
        # Compute current Q-value
        current_q = self.q_network(state).gather(1, action.unsqueeze(1)).squeeze()
        
        # Compute target Q-value
        with torch.no_grad():
            next_q = self.target_network(next_state).max(1)[0]
            target_q = reward + (1 - done) * self.gamma * next_q
        
        # Compute loss and optimize
        loss = F.mse_loss(current_q, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        # Update target network
        self.steps += 1
        if self.steps % self.target_update_freq == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())
        
        # Decay epsilon
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
        
        self.losses.append(loss.item())
        return loss.item()

def train_dqn_no_replay(agent, env, num_episodes=200):
    """Train DQN without replay buffer."""
    episode_rewards = []
    
    for episode in range(num_episodes):
        state, _ = env.reset()
        episode_reward = 0
        done = False
        
        while not done:
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # Train immediately (no replay)
            agent.train_step_immediate(state, action, reward, next_state, done)
            
            state = next_state
            episode_reward += reward
        
        episode_rewards.append(episode_reward)
    
    return episode_rewards

print("Training DQN WITHOUT experience replay...")
agent_no_replay = DQNAgentNoReplay(state_dim, action_dim, device)
rewards_no_replay = train_dqn_no_replay(agent_no_replay, env, num_episodes=200)
print(f"Final avg reward (last 50 eps): {np.mean(rewards_no_replay[-50:]):.2f}")

### Ablation 2: No Target Network

Use the same network for both current Q-values and target Q-values.

In [ ]:
class DQNAgentNoTarget(DQNAgent):
    """DQN without target network - uses Q-network for both current and target."""
    
    def train_step(self):
        """Train step without separate target network."""
        if len(self.replay_buffer) < self.batch_size:
            return None
        
        # Sample batch
        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)
        
        states = states.to(self.device)
        actions = actions.to(self.device)
        rewards = rewards.to(self.device)
        next_states = next_states.to(self.device)
        dones = dones.to(self.device)
        
        # Compute current Q-values
        current_q_values = self.q_network(states).gather(1, actions.unsqueeze(1)).squeeze()
        
        # Compute target using SAME network (no target network!)
        with torch.no_grad():
            next_q_values = self.q_network(next_states).max(1)[0]  # Using q_network, not target_network
            target_q_values = rewards + (1 - dones) * self.gamma * next_q_values
        
        # Compute loss and optimize
        loss = F.mse_loss(current_q_values, target_q_values)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        # Decay epsilon
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
        
        self.losses.append(loss.item())
        return loss.item()

print("Training DQN WITHOUT target network...")
agent_no_target = DQNAgentNoTarget(state_dim, action_dim, device)
rewards_no_target, _ = train_dqn(agent_no_target, env, num_episodes=200, print_every=100)
print(f"Final avg reward (last 50 eps): {np.mean(rewards_no_target[-50:]):.2f}")

### Compare All Variants

In [ ]:
# Compute moving averages for smoother comparison
window = 20
rewards_dqn_smooth = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
rewards_no_replay_smooth = np.convolve(rewards_no_replay, np.ones(window)/window, mode='valid')
rewards_no_target_smooth = np.convolve(rewards_no_target, np.ones(window)/window, mode='valid')

# Plot comparison
plt.figure(figsize=(12, 6))
plt.plot(rewards_dqn_smooth, label='DQN (full)', linewidth=2, color='green')
plt.plot(rewards_no_replay_smooth, label='DQN without replay', linewidth=2, color='red', linestyle='--')
plt.plot(rewards_no_target_smooth, label='DQN without target network', linewidth=2, color='orange', linestyle='-.')
plt.axhline(y=500, color='gray', linestyle='--', alpha=0.5, label='Goal')
plt.xlabel('Episode', fontsize=12)
plt.ylabel('Average Reward (20-episode window)', fontsize=12)
plt.title('DQN Ablation Study: Impact of Key Components', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print final performance
print("\nFinal Performance (last 50 episodes):")
print(f"  Full DQN:               {np.mean(episode_rewards[-50:]):.2f}")
print(f"  No experience replay:   {np.mean(rewards_no_replay[-50:]):.2f}")
print(f"  No target network:      {np.mean(rewards_no_target[-50:]):.2f}")

### Key Observations from Ablations

**Without Experience Replay**: Training is much less stable. Correlated samples lead to oscillations and slower convergence.

**Without Target Network**: Can still learn but training is less stable. The moving target problem causes higher variance in updates.

**Full DQN**: Combining both techniques yields the most stable and efficient learning. This is why DQN was a breakthrough!

## 11. Hyperparameter Sensitivity

Let's explore how key hyperparameters affect learning.

### Effect of Learning Rate

In [ ]:
def quick_train(lr, num_episodes=150):
    """Quick training run with specific learning rate."""
    agent = DQNAgent(state_dim, action_dim, device, lr=lr)
    rewards, _ = train_dqn(agent, env, num_episodes=num_episodes, print_every=1000)
    return rewards

print("Testing different learning rates...\n")
learning_rates = [1e-4, 1e-3, 1e-2]
lr_results = {}

for lr in learning_rates:
    print(f"Training with lr={lr}...")
    rewards = quick_train(lr, num_episodes=150)
    lr_results[lr] = rewards
    print(f"  Final avg (last 50): {np.mean(rewards[-50:]):.2f}\n")

In [ ]:
# Plot learning rate comparison
plt.figure(figsize=(12, 6))

for lr, rewards in lr_results.items():
    smooth_rewards = np.convolve(rewards, np.ones(20)/20, mode='valid')
    plt.plot(smooth_rewards, label=f'lr={lr}', linewidth=2)

plt.axhline(y=500, color='gray', linestyle='--', alpha=0.5, label='Goal')
plt.xlabel('Episode', fontsize=12)
plt.ylabel('Average Reward (20-episode window)', fontsize=12)
plt.title('Impact of Learning Rate on DQN Training', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Learning Rate Insights

- **Too small (1e-4)**: Slow learning, may not converge within 300 episodes
- **Just right (1e-3)**: Stable and efficient learning
- **Too large (1e-2)**: Can be unstable, may overshoot optimal Q-values

Like supervised learning, learning rate is the most important hyperparameter to tune!

## 12. Visualizing Q-Values

Let's peek inside the network to see what Q-values it learned.

In [ ]:
def visualize_q_values(agent, num_samples=100):
    """Visualize Q-values across different states."""
    # Collect states and Q-values during a few episodes
    states_list = []
    q_values_list = []
    
    state, _ = env.reset()
    for _ in range(num_samples):
        states_list.append(state)
        
        # Get Q-values
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_vals = agent.q_network(state_tensor).cpu().numpy()[0]
            q_values_list.append(q_vals)
        
        # Take action and continue
        action = agent.select_action(state, greedy=True)
        next_state, _, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            state, _ = env.reset()
        else:
            state = next_state
    
    states = np.array(states_list)
    q_values = np.array(q_values_list)
    
    # Plot Q-values for each action
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Q-values over time
    axes[0].plot(q_values[:, 0], label='Q(s, left)', linewidth=2)
    axes[0].plot(q_values[:, 1], label='Q(s, right)', linewidth=2)
    axes[0].fill_between(range(len(q_values)), q_values[:, 0], q_values[:, 1], alpha=0.2)
    axes[0].set_xlabel('Step', fontsize=11)
    axes[0].set_ylabel('Q-Value', fontsize=11)
    axes[0].set_title('Q-Values During Episode', fontsize=13, fontweight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)
    
    # Q-value distributions
    axes[1].hist(q_values[:, 0], bins=30, alpha=0.6, label='Q(s, left)', color='blue')
    axes[1].hist(q_values[:, 1], bins=30, alpha=0.6, label='Q(s, right)', color='orange')
    axes[1].set_xlabel('Q-Value', fontsize=11)
    axes[1].set_ylabel('Frequency', fontsize=11)
    axes[1].set_title('Q-Value Distributions', fontsize=13, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nQ-Value Statistics:")
    print(f"  Mean Q(s, left):  {q_values[:, 0].mean():.2f}")
    print(f"  Mean Q(s, right): {q_values[:, 1].mean():.2f}")
    print(f"  Max Q-value:      {q_values.max():.2f}")
    print(f"  Min Q-value:      {q_values.min():.2f}")

visualize_q_values(agent, num_samples=200)

### Interpreting Q-Values

**Q-values represent expected cumulative reward**: In CartPole, good Q-values should be close to 500 (max episode length).

**Action selection**: At each step, the agent picks the action with higher Q-value.

**Q-value differences**: When both actions have similar Q-values, the choice matters less. Large differences indicate clear preferences.

## 13. Key Takeaways

### Core Concepts

✓ **Why Neural Networks?** Tabular Q-learning fails for large/continuous state spaces. Neural networks enable **function approximation** and **generalization**.

✓ **Experience Replay**: Store transitions in a buffer and sample randomly. This breaks correlation, improves data efficiency, and stabilizes training.

✓ **Target Network**: Freeze a copy of the Q-network to provide stable TD targets. Update it periodically to prevent the "moving target" problem.

✓ **DQN = Q-learning + Neural Networks + Replay + Target Network**: Each component is crucial for stable deep RL.

### The DQN Algorithm

```
1. Initialize Q-network and target network
2. For each step:
   a. Select action using ε-greedy
   b. Execute action, observe reward and next state
   c. Store transition in replay buffer
   d. Sample random batch from buffer
   e. Compute TD target using target network
   f. Update Q-network via gradient descent
   g. Periodically update target network
```

### Extensions and Improvements

DQN has inspired many improvements:
- **Double DQN**: Reduces overestimation bias
- **Dueling DQN**: Separate value and advantage streams
- **Prioritized Experience Replay**: Sample important transitions more often
- **Rainbow DQN**: Combines multiple improvements
- **Distributional RL**: Model full return distribution, not just mean

### When to Use DQN

**Good for**:
- Discrete action spaces
- High-dimensional state spaces (images)
- Off-policy learning (can learn from past data)

**Not ideal for**:
- Continuous action spaces (use DDPG, TD3, or SAC instead)
- Very sparse rewards (may need reward shaping)
- Real-time learning (requires large replay buffer)

### The Bigger Picture

DQN (2015) was a landmark achievement - the first deep RL algorithm to learn from pixels on Atari games at superhuman level. It bridged the gap from tabular RL to modern deep RL, enabling:
- AlphaGo (game playing)
- Robotics control
- Autonomous systems
- Resource optimization

**You now understand the foundation of modern deep reinforcement learning!**

## 14. Experiments to Try

1. **Different environments**: Try DQN on other Gym environments like `MountainCar-v0` or `LunarLander-v2`

2. **Hyperparameter tuning**: Experiment with batch size, buffer size, target update frequency

3. **Network architecture**: Try deeper/wider networks or add dropout

4. **Epsilon schedule**: Test different exploration strategies (linear decay, exponential, etc.)

5. **Double DQN**: Implement the Double DQN improvement to reduce overestimation

6. **Prioritized replay**: Weight important transitions higher when sampling

7. **Frame stacking**: For Atari games, stack 4 frames to capture motion

8. **Compare to random policy**: Quantify improvement over baseline

## Summary

Congratulations! You've learned:

- ✓ Why tabular Q-learning fails for complex state spaces
- ✓ How neural networks approximate Q-functions
- ✓ The experience replay mechanism and its benefits
- ✓ The target network trick for stable training
- ✓ Complete DQN implementation from scratch
- ✓ How to train and evaluate DQN agents
- ✓ The importance of each component through ablations

**DQN is the foundation of modern deep RL** - master this, and you're ready for advanced algorithms like PPO, SAC, and more!

**Next steps**: Explore policy gradient methods (e.g., REINFORCE, Actor-Critic, PPO) which directly optimize the policy instead of Q-values.